In [ ]:
"""
Texas Rangers 2026 시즌 데이터 수집 스크립트
=============================================
Baseball Savant (Statcast) 기반으로 Rangers 경기 데이터를 수집하고
EDA용 CSV 파일을 생성합니다.

사용법:
    python collect_rangers_2026.py

업데이트 방법:
    START_DATE / END_DATE 변수만 수정 후 재실행하면 덮어씌워집니다.

출력 파일:
    rangers_batter_gamelogs_2026.csv
    rangers_pitcher_gamelogs_2026.csv
    rangers_hitting_statcast_2026.csv
    rangers_hitting_batted_2026.csv
    rangers_hitting_discipline_2026.csv
    rangers_pitching_statcast_2026.csv
    rangers_pitching_batted_2026.csv
    rangers_pitching_discipline_2026.csv
    rangers_roster_2026.csv
    rangers_2026_batters_daily_final.csv
    rangers_2026_pitchers_daily_final.csv
    data/league_2026/tex_statcast_2026.csv  (raw cache)

주의:
    - R / RBI / SB / CS 는 Statcast에서 직접 산출 불가 → 0으로 처리
    - 시즌 초반 소수 경기는 지표 신뢰도 낮음
    - 약 30초 소요 (Statcast 쿼리)
"""

import pandas as pd
import numpy as np
from pathlib import Path
from pybaseball import statcast, playerid_reverse_lookup
import warnings
warnings.filterwarnings('ignore')

# ── 설정 ──────────────────────────────────────────────────
START_DATE = "2026-03-27"   # 시즌 개막일
END_DATE   = "2026-04-11"   # 수집 종료일 (오늘 날짜로 업데이트)
OUT_DIR    = Path(__file__).parent
RAW_DIR    = OUT_DIR / "data" / "league_2026"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Statcast 다운로드 ──────────────────────────────────
print(f"[1/4] Statcast 다운로드 ({START_DATE} ~ {END_DATE})...")
raw_path = RAW_DIR / "tex_statcast_2026.csv"

all_data = statcast(START_DATE, END_DATE)
tex = all_data[(all_data['home_team']=='TEX') | (all_data['away_team']=='TEX')].copy()
tex['game_date'] = pd.to_datetime(tex['game_date'])
tex.to_csv(raw_path, index=False)
print(f"    TEX 투구수: {len(tex):,}  경기수: {tex['game_pk'].nunique()}")

# ── 2. 타자/투수 투구 분리 ────────────────────────────────
tex_bat = tex[
    ((tex['home_team']=='TEX') & (tex['inning_topbot']=='Bot')) |
    ((tex['away_team']=='TEX') & (tex['inning_topbot']=='Top'))
].copy()

tex_pit = tex[
    ((tex['home_team']=='TEX') & (tex['inning_topbot']=='Top')) |
    ((tex['away_team']=='TEX') & (tex['inning_topbot']=='Bot'))
].copy()

# 타자 이름 매핑 (MLBAM ID → 이름)
batter_ids = tex_bat['batter'].unique().tolist()
id_map = playerid_reverse_lookup(batter_ids, key_type='mlbam')
id_map['pname'] = (id_map['name_first'].str.capitalize() + ' ' +
                   id_map['name_last'].str.capitalize())
id_map = id_map.rename(columns={'key_mlbam':'batter'})[['batter','pname']]
tex_bat = tex_bat.merge(id_map, on='batter', how='left')

# 투수 이름 정리 (Last, First → First Last)
def fix_name(n):
    if pd.isna(n): return n
    parts = str(n).split(', ')
    return f"{parts[1]} {parts[0]}" if len(parts) == 2 else n

tex_pit['pname'] = tex_pit['player_name'].apply(fix_name)

# ── 3. 공통 유틸 함수 ─────────────────────────────────────
def is_barrel(ev, angle):
    return ((ev >= 98) & angle.between(26, 50))

def whiff_rate(grp):
    sw = grp['description'].isin(['swinging_strike','swinging_strike_blocked',
                                   'foul','foul_tip','hit_into_play']).sum()
    wh = grp['description'].isin(['swinging_strike','swinging_strike_blocked']).sum()
    return round(wh / sw * 100, 1) if sw > 0 else 0.0

def chase_rate(grp):
    out_zone = ~grp['zone'].isin(range(1, 10))
    swings   = grp['description'].isin(['swinging_strike','swinging_strike_blocked',
                                         'foul','foul_tip','hit_into_play'])
    oz = out_zone.sum()
    return round((out_zone & swings).sum() / oz * 100, 1) if oz > 0 else 0.0

def zone_rate(grp):
    iz = grp['zone'].isin(range(1, 10)).sum()
    return round(iz / len(grp) * 100, 1) if len(grp) > 0 else 0.0

def meatball_rate(grp):
    mb = (grp['zone'] == 5).sum()
    return round(mb / len(grp) * 100, 1) if len(grp) > 0 else 0.0

def zone_contact_rate(grp):
    iz_sw  = (grp['zone'].isin(range(1,10)) &
              grp['description'].isin(['swinging_strike','swinging_strike_blocked',
                                        'foul','foul_tip','hit_into_play'])).sum()
    iz_con = (grp['zone'].isin(range(1,10)) &
              grp['description'].isin(['foul','foul_tip','hit_into_play'])).sum()
    return round(iz_con / iz_sw * 100, 1) if iz_sw > 0 else 0.0

print("[2/4] 타자 파일 생성...")

# ── 4-A. 타자 게임로그 ────────────────────────────────────
def bat_game_stats(grp):
    ev = grp['events']
    h   = ev.isin(['single','double','triple','home_run']).sum()
    d2  = (ev=='double').sum()
    d3  = (ev=='triple').sum()
    hr  = (ev=='home_run').sum()
    bb  = ev.isin(['walk','intent_walk']).sum()
    so  = ev.isin(['strikeout','strikeout_double_play']).sum()
    hbp = (ev=='hit_by_pitch').sum()
    sf  = ev.isin(['sac_fly','sac_bunt']).sum()
    pa  = len(grp)
    ab  = pa - bb - hbp - sf
    tb  = (h - d2 - d3 - hr) + 2*d2 + 3*d3 + 4*hr
    avg = round(h/ab,   3) if ab > 0 else 0.0
    obp = round((h+bb+hbp)/pa, 3) if pa > 0 else 0.0
    slg = round(tb/ab,  3) if ab > 0 else 0.0
    return pd.Series({'PA':pa,'AB':ab,'H':int(h),'2B':int(d2),'3B':int(d3),
                      'HR':int(hr),'BB':int(bb),'SO':int(so),'HBP':int(hbp),
                      'game_AVG':avg,'game_OBP':obp,
                      'game_SLG':slg,'game_OPS':round(obp+slg,3)})

pa_df = tex_bat[tex_bat['events'].notna()].copy()
bat_gl = (pa_df.groupby(['batter','pname','game_date','home_team','away_team'])
          .apply(bat_game_stats, include_groups=False)
          .reset_index()
          .rename(columns={'pname':'name','game_date':'Date',
                           'home_team':'Home Tm','away_team':'Away Tm',
                           'batter':'player_id'}))
for c in ['R','RBI','SB','CS']:
    bat_gl[c] = 0  # Statcast에서 직접 산출 불가

col_order = ['player_id','name','Date','Home Tm','Away Tm',
             'PA','AB','R','H','2B','3B','HR','RBI','BB','SO','SB','CS','HBP',
             'game_AVG','game_OBP','game_SLG','game_OPS']
bat_gl[col_order].to_csv(OUT_DIR / 'rangers_batter_gamelogs_2026.csv', index=False)

# ── 4-B. 타자 Statcast 시즌 집계 ─────────────────────────
bip = tex_bat[tex_bat['launch_speed'].notna()].copy()
bip['barrel']   = is_barrel(bip['launch_speed'], bip['launch_angle'])
bip['hard_hit'] = bip['launch_speed'] >= 95
xslg_col = 'estimated_slg_using_speedangle'

pa_cnt = tex_bat[tex_bat['events'].notna()].groupby('pname').size().reset_index(name='PA')
hit_stat_rows = []
for name, grp in bip.groupby('pname'):
    hit_stat_rows.append({
        'Player': name,
        'wOBA':          round(grp['woba_value'].mean(), 3),
        'xwOBA':         round(grp['estimated_woba_using_speedangle'].mean(), 3),
        'xBA':           round(grp['estimated_ba_using_speedangle'].mean(), 3),
        'Exit Velocity': round(grp['launch_speed'].mean(), 1),
        'Barrel %':      round(grp['barrel'].mean() * 100, 1),
        'Hard Hit %':    round(grp['hard_hit'].mean() * 100, 1),
    })
hit_stat = pd.DataFrame(hit_stat_rows).merge(
    pa_cnt.rename(columns={'pname':'Player'}), on='Player', how='left')
hit_stat.to_csv(OUT_DIR / 'rangers_hitting_statcast_2026.csv', index=False)

# ── 4-C. 타자 타구 방향 ───────────────────────────────────
bb_map = {'ground_ball':'GB','fly_ball':'FB','line_drive':'LD','popup':'PU'}
tex_bat_bb = tex_bat[tex_bat['bb_type'].notna()].copy()
tex_bat_bb['bb_c'] = tex_bat_bb['bb_type'].map(bb_map)
bb_agg = (tex_bat_bb.groupby(['pname','bb_c']).size()
          .unstack(fill_value=0).reset_index())
for c in ['GB','FB','LD','PU']:
    if c not in bb_agg.columns: bb_agg[c] = 0
bb_agg['total'] = bb_agg[['GB','FB','LD','PU']].sum(axis=1)
for c in ['GB','FB','LD']:
    bb_agg[f'{c} %'] = (bb_agg[c] / bb_agg['total'] * 100).round(1)
bb_agg.rename(columns={'pname':'Player'})[['Player','GB %','FB %','LD %']].to_csv(
    OUT_DIR / 'rangers_hitting_batted_2026.csv', index=False)

# ── 4-D. 타자 플레이트 디시플린 ──────────────────────────
dis_rows = []
for name, grp in tex_bat.groupby('pname'):
    dis_rows.append({'Player': name,
                     'Zone %':         zone_rate(grp),
                     'Chase %':        chase_rate(grp),
                     'Whiff %':        whiff_rate(grp),
                     'Meatball %':     meatball_rate(grp),
                     'Zone Contact %': zone_contact_rate(grp)})
pd.DataFrame(dis_rows).to_csv(OUT_DIR / 'rangers_hitting_discipline_2026.csv', index=False)

# ── 4-E. 타자 daily ───────────────────────────────────────
def bat_daily(grp):
    b = grp[grp['launch_speed'].notna()]
    sw = grp['description'].isin(['swinging_strike','swinging_strike_blocked',
                                   'foul','foul_tip','hit_into_play'])
    wh = grp['description'].isin(['swinging_strike','swinging_strike_blocked'])
    pa = grp['events'].notna().sum()
    n  = len(b)
    br = ((b['launch_speed']>=98) & b['launch_angle'].between(26,50)).sum()
    hh = (b['launch_speed']>=95).sum()
    ss = b['launch_angle'].between(8,32).sum()
    return pd.Series({
        'PA': pa, 'Pitches_seen': len(grp),
        'EV':    round(b['launch_speed'].mean(),2) if n>0 else np.nan,
        'Max_EV':round(b['launch_speed'].max(),1)  if n>0 else np.nan,
        'LA':    round(b['launch_angle'].mean(),3) if n>0 else np.nan,
        'Barrel':int(br), 'Hard_Hit':int(hh), 'Sweet_Spot':int(ss), 'Total_BIP':n,
        'xBA':   round(b['estimated_ba_using_speedangle'].mean(),3) if n>0 else np.nan,
        'xSLG':  round(b[xslg_col].mean(),3) if n>0 and xslg_col in b and b[xslg_col].notna().any() else np.nan,
        'xwOBA': round(b['estimated_woba_using_speedangle'].mean(),3) if n>0 else np.nan,
        'Whiff': int(wh.sum()), 'Swing': int(sw.sum()),
        'Bat_Speed':    round(grp['bat_speed'].mean(),2)    if 'bat_speed'    in grp and grp['bat_speed'].notna().any()    else np.nan,
        'Swing_Length': round(grp['swing_length'].mean(),2) if 'swing_length' in grp and grp['swing_length'].notna().any() else np.nan,
        'Attack_Angle': round(grp['attack_angle'].mean(),2) if 'attack_angle' in grp and grp['attack_angle'].notna().any() else np.nan,
        'Barrel_pct':     round(br/n*100,1) if n>0 else 0.0,
        'Hard_Hit_pct':   round(hh/n*100,1) if n>0 else 0.0,
        'Sweet_Spot_pct': round(ss/n*100,1) if n>0 else 0.0,
        'Whiff_pct': round(wh.sum()/sw.sum()*100,1) if sw.sum()>0 else 0.0,
    })

(tex_bat.groupby(['batter','pname','game_date'])
 .apply(bat_daily, include_groups=False)
 .reset_index()
 .rename(columns={'pname':'player_name','game_date':'game_date'})
 .to_csv(OUT_DIR / 'rangers_2026_batters_daily_final.csv', index=False))

print("[3/4] 투수 파일 생성...")

# ── 5-A. 투수 게임로그 ────────────────────────────────────
pit_pa = tex_pit[tex_pit['events'].notna()].copy()

def pit_game_stats(grp):
    ev = grp['events']
    h   = ev.isin(['single','double','triple','home_run']).sum()
    hr  = (ev=='home_run').sum()
    bb  = ev.isin(['walk','intent_walk']).sum()
    so  = ev.isin(['strikeout','strikeout_double_play']).sum()
    hbp = (ev=='hit_by_pitch').sum()
    return pd.Series({'PA':len(grp),'H':int(h),'HR':int(hr),
                      'BB':int(bb),'SO':int(so),'HBP':int(hbp),'ER':0})

(pit_pa.groupby(['pitcher','pname','game_date','home_team','away_team'])
 .apply(pit_game_stats, include_groups=False)
 .reset_index()
 .rename(columns={'pname':'name','game_date':'Date',
                  'home_team':'Home Tm','away_team':'Away Tm',
                  'pitcher':'player_id'})
 .to_csv(OUT_DIR / 'rangers_pitcher_gamelogs_2026.csv', index=False))

# ── 5-B. 투수 Statcast 시즌 집계 ─────────────────────────
bip_p = tex_pit[tex_pit['launch_speed'].notna()].copy()
bip_p['barrel']   = is_barrel(bip_p['launch_speed'], bip_p['launch_angle'])
bip_p['hard_hit'] = bip_p['launch_speed'] >= 95
pa_cnt_p = tex_pit[tex_pit['events'].notna()].groupby('pname').size().reset_index(name='PA')

pit_stat_rows = []
for name, grp in bip_p.groupby('pname'):
    pit_stat_rows.append({
        'Player': name,
        'wOBA':          round(grp['woba_value'].mean(), 3),
        'xwOBA':         round(grp['estimated_woba_using_speedangle'].mean(), 3),
        'xBA':           round(grp['estimated_ba_using_speedangle'].mean(), 3),
        'Exit Velocity': round(grp['launch_speed'].mean(), 1),
        'Barrel %':      round(grp['barrel'].mean() * 100, 1),
        'Hard Hit %':    round(grp['hard_hit'].mean() * 100, 1),
    })
pd.DataFrame(pit_stat_rows).merge(
    pa_cnt_p.rename(columns={'pname':'Player'}), on='Player', how='left'
).to_csv(OUT_DIR / 'rangers_pitching_statcast_2026.csv', index=False)

# ── 5-C. 투수 타구 방향 ───────────────────────────────────
tex_pit_bb = tex_pit[tex_pit['bb_type'].notna()].copy()
tex_pit_bb['bb_c'] = tex_pit_bb['bb_type'].map(bb_map)
pit_bb = (tex_pit_bb.groupby(['pname','bb_c']).size()
          .unstack(fill_value=0).reset_index())
for c in ['GB','FB','LD','PU']:
    if c not in pit_bb.columns: pit_bb[c] = 0
pit_bb['total'] = pit_bb[['GB','FB','LD','PU']].sum(axis=1)
for c in ['GB','FB','LD']:
    pit_bb[f'{c} %'] = (pit_bb[c] / pit_bb['total'] * 100).round(1)
pit_bb.rename(columns={'pname':'Player'})[['Player','GB %','FB %','LD %']].to_csv(
    OUT_DIR / 'rangers_pitching_batted_2026.csv', index=False)

# ── 5-D. 투수 플레이트 디시플린 ──────────────────────────
pit_dis_rows = []
for name, grp in tex_pit.groupby('pname'):
    pit_dis_rows.append({'Player': name,
                         'Zone %':         zone_rate(grp),
                         'Chase %':        chase_rate(grp),
                         'Whiff %':        whiff_rate(grp),
                         'Meatball %':     meatball_rate(grp),
                         'Zone Contact %': zone_contact_rate(grp)})
pd.DataFrame(pit_dis_rows).to_csv(OUT_DIR / 'rangers_pitching_discipline_2026.csv', index=False)

# ── 5-E. 투수 daily ───────────────────────────────────────
def pit_daily(grp):
    b  = grp[grp['launch_speed'].notna()]
    sw = grp['description'].isin(['swinging_strike','swinging_strike_blocked',
                                   'foul','foul_tip','hit_into_play'])
    wh = grp['description'].isin(['swinging_strike','swinging_strike_blocked'])
    ev = grp['events']
    k  = ev.isin(['strikeout','strikeout_double_play']).sum()
    bb = ev.isin(['walk','intent_walk']).sum()
    outs = ev.isin(['strikeout','strikeout_double_play','field_out','force_out',
                    'grounded_into_double_play','double_play','triple_play',
                    'fielders_choice_out','sac_fly','sac_bunt']).sum()
    n = len(b)
    br = ((b['launch_speed']>=98) & b['launch_angle'].between(26,50)).sum()
    hh = (b['launch_speed']>=95).sum()
    return pd.Series({
        'Pitches_thrown': len(grp),
        'Avg_Velo': round(grp['release_speed'].mean(),3) if grp['release_speed'].notna().any() else np.nan,
        'Max_Velo': round(grp['release_speed'].max(),1)  if grp['release_speed'].notna().any() else np.nan,
        'Avg_Spin': round(grp['release_spin_rate'].mean(),2) if grp['release_spin_rate'].notna().any() else np.nan,
        'EV_allowed':      round(b['launch_speed'].mean(),2) if n>0 else np.nan,
        'Barrel_allowed':  int(br), 'Hard_Hit_allowed': int(hh), 'Total_BIP': n,
        'xBA_against':     round(b['estimated_ba_using_speedangle'].mean(),3)    if n>0 else np.nan,
        'xwOBA_against':   round(b['estimated_woba_using_speedangle'].mean(),3)  if n>0 else np.nan,
        'Whiff': int(wh.sum()), 'Swing': int(sw.sum()),
        'K': int(k), 'BB': int(bb), 'Outs': int(outs),
        'V_Break': round(grp['pfx_z'].mean()*-12,2) if grp['pfx_z'].notna().any() else np.nan,
        'H_Break': round(grp['pfx_x'].mean()*12,2)  if grp['pfx_x'].notna().any() else np.nan,
        'Release_Extension': round(grp['release_extension'].mean(),2) if grp['release_extension'].notna().any() else np.nan,
        'IP_float':              round(int(outs//3) + (outs%3)/3, 3),
        'Whiff_pct':             round(wh.sum()/sw.sum()*100,1) if sw.sum()>0 else 0.0,
        'Hard_Hit_allowed_pct':  round(hh/n*100,1) if n>0 else 0.0,
    })

(tex_pit.groupby(['pitcher','pname','game_date'])
 .apply(pit_daily, include_groups=False)
 .reset_index()
 .rename(columns={'pname':'player_name'})
 .to_csv(OUT_DIR / 'rangers_2026_pitchers_daily_final.csv', index=False))

# ── 6. 로스터 ─────────────────────────────────────────────
print("[4/4] 로스터 생성...")
batters  = tex_bat['pname'].dropna().unique().tolist()
pitchers = tex_pit['pname'].dropna().unique().tolist()
roster = ([{'name':n,'type':'batter'}  for n in batters] +
          [{'name':n,'type':'pitcher'} for n in pitchers])
pd.DataFrame(roster).to_csv(OUT_DIR / 'rangers_roster_2026.csv', index=False)

print("\n완료!")
print(f"  타자: {len(batters)}명 / 투수: {len(pitchers)}명")
print(f"  수집 기간: {START_DATE} ~ {END_DATE}")
print(f"  저장 경로: {OUT_DIR}")
